<!-- [intro-a] -->

# ParallelProse — v2, Comparing Two Books (Intro)

v1's loop retrieved, wrote, and judged an answer about *one* book. v2 asks it to do the same
job across *two* — comparing what they each say about the same issue. That single change, one
book to two, is the root of everything new here.

This notebook walks through the new concepts that change forces into existence, one at a time,
staying at arm's length from code and field names. `v2-02` opens each one up mechanically once
the shape here is clear.

<!-- [A-a] -->

## A — The big picture

Same loop, now watching two books side by side instead of one.

<!-- [A.1a] -->

```mermaid
flowchart LR
    A[("Book A")] --> L
    B[("Book B")] --> L
    subgraph L["the loop — unchanged shape"]
        direction TB
        RET["retriever"] --> COMP["composer"]
        COMP --> REF["reflect"]
        REF -.->|"retry, per book"| RET
    end
    REF -->|"good enough"| ANS["comparison answer"]
```

<!-- [A.2a] -->

The three boxes inside the loop are the exact three from v1 — `retriever`, `composer`,
`reflect`. Nothing there moved. What changed is that each one now has to think about two books
instead of one, and that's what sections B through G are each about.

<!-- [B-a] -->

## B — The state that carries it all

Before either book gets touched, there has to be somewhere to keep two books' worth of notes
without mixing them up.

<!-- [B.1a] -->

```mermaid
flowchart LR
    S["the loop's shared state"] --> SA["Book A's slot"]
    S --> SB["Book B's slot"]
```

<!-- [B.2a] -->

Picture the loop's memory in v1 as a single scratchpad — one book, one running note of what's
been retrieved, one verdict. v2 can't get away with one scratchpad for two books: whatever's
true of Book A's coverage has nothing to do with Book B's, and mixing them means losing track of
which book actually said what. So the scratchpad grows a second, parallel compartment — one
labeled slot per book, still one scratchpad.

Every piece from here on reads from and writes into one of these two slots — that's what "per
book" means for the rest of this notebook.

<!-- [C-a] -->

## C — Getting the question to both books

The same sentence doesn't always mean the same thing to two different books.

<!-- [C.1a] -->

```mermaid
flowchart LR
    Q["one question"] --> SPLIT["rephrase, per book"]
    SPLIT --> QA["question, in Book A's terms"]
    SPLIT --> QB["question, in Book B's terms"]
    QA --> SA["Book A's slot"]
    QB --> SB["Book B's slot"]
```

<!-- [C.2a] -->

In v1 there was only ever one question, asked one way, of one book. Two books means the same
question can land very differently — a modern word may simply not exist in an old book's
vocabulary at all (the recurring example here: asking about "virtue" gets nothing back from a
book that only ever calls it "arete").

Two new pieces handle this, from two different angles: one rephrases the question in whatever
way suits each book generally (query-splitting); the other specifically swaps in a book's own
period-appropriate term for the same idea (terminology-bridging) — reword broadly vs. translate
one known term precisely.

Each book now holds its own tailored version of the question, sitting in its slot from section
B. That's what section D actually goes and searches with.

<!-- [D-a] -->

## D — Choosing how to retrieve, per book

v1 always searched the same way, no matter the book or the question.

<!-- [D.1a] -->

```mermaid
flowchart TB
    subgraph TOOLBOX["the same 4 strategies from v1 — no new ones"]
        direction LR
        P["parent_retriever"]
        S["self-query"]
        B["bm25"]
        E["ensemble"]
    end
    QA["Book A's question"] --> TOOLBOX --> RA["Book A's material"]
    QB["Book B's question"] --> TOOLBOX --> RB["Book B's material"]
```

<!-- [D.2a] -->

v1's retriever never actually chose anything — it always called the same one strategy, on the
same one book. v2 doesn't add new strategies; the 4 from v1 are still exactly what's on offer.
What's new is that something now *picks* among them, per book, per question — because what works
best for searching one book (a dense reference structure, say) may not be what works for another
(a text that needs semantic matching more than keyword matching).

Whatever comes back for each book's slot — good, thin, or empty — is exactly what section E has
to judge next.

<!-- [E-a] -->

## E — Judging what came back, per book

A single yes/no can't say enough about two separate books.

<!-- [E.1a] -->

```mermaid
flowchart LR
    RA["Book A's material"] --> REF{{"reflect"}}
    RB["Book B's material"] --> REF
    REF -->|"retrieval miss"| M1["worth trying again"]
    REF -->|"wrong words"| M2["worth rewording / bridging a term"]
    REF -->|"genuine silence"| M3["say so — no retry helps"]
```

<!-- [E.2a] -->

v1's `reflect` asked one question of one answer: good enough, yes or no. v2 has to ask that once
per book, and a plain yes/no stops being useful on its own — "not good enough" doesn't say what
to actually do about it. So the verdict grows from a gate into a diagnosis: did retrieval simply
miss it (worth trying again), did the question not speak the book's language (worth rewording or
bridging a term), or does the book truly never address this at all (no retry will help — that's
an answer in itself, not a failure)?

This per-book diagnosis is what decides two different things next: what `composer` is allowed to
say about that book (section F), and whether that book's slot goes back through C/D at all
(section G).

<!-- [F-a] -->

## F — Writing the comparison

Comparing two books needs a shape that shows both of them, not one blended paragraph.

<!-- [F.1a] -->

```mermaid
flowchart TB
    D["composer's draft"] --> AG["what they agree on"]
    D --> DIS["what they disagree on"]
    D --> UA["unique to Book A"]
    D --> UB["unique to Book B"]
```

<!-- [F.2a] -->

v1's `composer` wrote one paragraph, answering one question about one book — there was nothing
to compare, so nothing needed separating out. v2's `composer` is handed two books' worth of
material and has to write an actual comparison, so its draft gets explicit slots instead of one
blended narrative: where the books agree, where they diverge, what only shows up in one of them.

When section E already ruled a book "genuinely silent" on the topic, `composer` says exactly
that as one of the findings ("Book A never addresses this; Book B argues X") instead of writing
an apologetic paragraph that reads like the system failed.

This structured shape is also what makes E's per-book judgment checkable in the first place — a
free-form paragraph can't be graded book-by-book, but explicit slots can. That's the loop back to
E: judging and writing depend on each other.

<!-- [G-a] -->

## G — Closing the loop precisely

If only one book came back thin, only that book should have to try again.

<!-- [G.1a] -->

```mermaid
flowchart LR
    REF{{"reflect"}} -->|"Book A flagged, Book B fine"| RETRY["retry: only Book A"]
    RETRY --> C_["back to C — reword Book A's question"]
    C_ --> D_["back to D — search Book A again"]
    D_ --> REF
```

<!-- [G.2a] -->

v1 had one retry lever — the whole loop reran, or it didn't. v2's `reflect` already knows, per
book, whether a retry would even help (section E), so retry only has to touch the book that was
actually flagged. Book A gets sent back through rewording (C) and searching (D) with a narrowed
question; Book B's already-good material sits untouched in its slot from section B.

This is genuinely a rerun of A through D for one book, not a new mechanism — the same loop, aimed
narrowly. The retried book's fresh material goes straight back to E for another look, and
eventually to F once both books are settled.

<!-- [close-a] -->

## Recap

One book becoming two forced six new pieces into a loop that otherwise didn't change shape:

- **B** — one scratchpad, two labeled slots.
- **C** — the same question, reworded and term-bridged per book.
- **D** — a choice among v1's existing 4 strategies, made per book instead of hardcoded.
- **E** — `reflect`'s yes/no becomes a diagnosis: miss, mismatch, or genuine silence.
- **F** — `composer`'s one paragraph becomes agreement / disagreement / unique-to-each, with
  silence stated as a finding.
- **G** — retry aimed at only the flagged book, not the whole loop.

Next: `v2-02` — the mechanics behind each of these (the actual state shape, the diagnostic
label's values, how the tool-selecting retriever is wired, what "structured" means for
`composer`'s output type).